In [1]:
import sys
import pandas as pd
import numpy as np
import navis
import pymaid
import random
import re
import ast
import json


import skeletor as sk
import cloudvolume as cv
import fafbseg
import caveclient
import k3d

D:\flywire_backup\cave\cave\lib\site-packages\blessed\terminal.py:183: UserWarning: Failed to setupterm(kind='xterm-color'): Could not find terminal xterm-color
  warnings.warn(msg)


In [20]:
#megalopta project id: 8
#megalopta test project id: 25
#megalopta flywire project id: 28
#eciton CX: 7
path = 'D:/flywire_backup/cave/cat_client.json' # replace with your path

with open(path, 'r') as f:
    conn_file = json.load(f)
api_token = conn_file['api_token']
http_user = conn_file['http_user']
http_password = conn_file['http_password']
server = conn_file['server']

cat_client = pymaid.CatmaidInstance(server = server, 
                                api_token = api_token,
                                caching = True,
                                project_id = 8,
                                http_user = http_user,
                                http_password = http_password 
                                )

client = caveclient.CAVEclient(server_address='https://global.connectomics.braininbrain.org')

client = caveclient.CAVEclient(datastack_name='megalopta_fb_eb_datastack')
auth = client.auth
cg = client.chunkedgraph
print('client set')


################################ other useful functions
# client.info.get_datastacks()
#print(f"My current token is: {auth.token}")

INFO  : Global CATMAID instance set. Caching is ON. (pymaid)


client set


In [4]:
# ngl_segmentation = 'graphene://https://local.cave.braininbrain.org/segmentation/table/heinze_eciton_PB_v1' # obtain from CAVE 
ngl_segmentation = 'graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_FB_EB_v3'
navis.patch_cloudvolume()
vol = cv.CloudVolume(ngl_segmentation, use_https=True, progress=False)

INFO  : cloud-volume successfully patched! (navis)


In [15]:
# # Example for nueron with a single segment id: PFN_LY_L4_AT_MS_VN2_PFN2E
# pfn1 = [576460752498180584]

# # Example for nueron with two segment ids: PFN_LY_L4_MS_VN3_PFN2E
# pfn2 = [576460752505789669, 576460752436280917]

# namelist = ['PFN_LY_L4_AT_MS_VN2_PFN2E_CAVE', 'PFN_LY_L4_MS_VN3_PFN2E_CAVE']

# pfn1_m = vol.mesh.get(pfn1, remove_duplicate_vertices=True, as_navis=True) # download neuron mesh
# pfn2_m = vol.mesh.get(pfn2, remove_duplicate_vertices=True, fuse=True, allow_missing=True, as_navis=True)

# pfn1_m = navis.NeuronList(pfn1_m)
# pfn2_m = navis.NeuronList(pfn2_m)

# m = pfn1_m + pfn2_m # m = 'mesh'

# for neuron, name in zip(m, namelist): # name the neurons (skids are better)
#     neuron.name = name
    
# m

,type,name,id,units,n_vertices,n_faces
0,navis.MeshNeuron,PFN_LY_L4_AT_MS_VN2_PFN2E_CAVE,576460752498180584,1 nanometer,84701,167237
1,navis.MeshNeuron,PFN_LY_L4_MS_VN3_PFN2E_CAVE,NA,1 nanometer,84395,166800


In [5]:
er = [576460753128455242]
er = vol.mesh.get(er, remove_duplicate_vertices=True, as_navis=True) # download neuron mesh
er = navis.NeuronList(er)

In [6]:
er[0].name = 'ER_L_MBUd_125523'

In [7]:
er

,type,name,id,units,n_vertices,n_faces
0,navis.MeshNeuron,ER_L_MBUd_125523,576460753128455242,1 nanometer,291635,582292


In [8]:
mesh_list = er.copy() # back up just in case
        
# cave meshes are very complex, so simplify
m_simp = navis.simplify_mesh(mesh_list, 0.3, backend='pyfqmr', inplace=False, parallel=True, progress=True) #lower value, more simple

Simplifying:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# preview simplified mesh
fig=navis.plot3d(m_simp)

In [10]:
'''SKELETONIZE 
- this is the tricky part...there are several options to skeletonize the mesh, 
but I haven't solved this. Below typically works for visualization purposes,
but results in spikey, convoluted skeletons that *USUALLY* need to be cleaned before being used for 
morphological analysis (nblast, etc)...however the eciton examples below look quite nice. You might 
be able to use this exactly as is! 

- for more, see https://navis-org.github.io/skeletor/
- see: https://www.caveconnecto.me/pcg_skel/
- see: https://meshparty.readthedocs.io/en/latest/
- play around with: client.skeleton.get_bulk_skeletons()

below uses skeletor:
'''
skel_list = []

for mesh in m_simp:
    sk = navis.skeletonize(mesh)
    skel_list.append(sk)

skel_list = navis.NeuronList(skel_list)

# then join aka 'heal' unjoined fragments
sk_h_list = []

for idx, neuron in enumerate(skel_list, start=1):
    if len(neuron.root) > 1: # number of disconnected fragments
        skel_h = navis.heal_skeleton(neuron)
        skel_h.soma = None
        sk_h_list.append(skel_h)
        print(f'\r{idx}/{len(skel_list)} skeleton healed')
        sys.stdout.flush()

for idx, neuron in enumerate(sk_h_list, start=1):
    neuron.nodes['radius'] = neuron.nodes['radius'].replace(0, 50) # smallest node size shouldn't be 0. 50 is arbitrary

sk_h_list = navis.NeuronList(sk_h_list)

sk_h_list

1/1 skeleton healed


,type,name,id,n_nodes,n_connectors,n_branches,n_leafs,cable_length,soma,units
0,navis.TreeNeuron,ER_L_MBUd_125523,576460753128455242,7543,None,572,630,2.178195e+06,None,1 nanometer


In [ ]:
# preview skeletonized neuron with node radius 
fig=navis.plot3d(sk_h_list, radius=True)

In [ ]:
# preview skeletonized neuron with node radius off 
fig=navis.plot3d(sk_h_list, radius=False)

In [21]:
# Will stop here, but you can then download Eciton catmaid neurons, prune so it's just the main branch:
# mainFiber = navis.longest_neurite(cat_neuron)
# then stich with your skeletonized mesh
# to_stitch = neuron1 + neuron2
# complete_n = navis.stitch_skeletons(to_stitch, method=[0,10], master='LARGEST')

# upload skeletonized neurons to catmaid
pymaid.upload_neuron(sk_h_list, import_connectors=False, remote_instance=cat_client)

Uploading:   0%|          | 0/1 [00:00<?, ?it/s]

{576460753128455242: {'neuron_id': 508950,
  'skeleton_id': 508949,
  'node_id_map': {0: 8861710,
   8323: 8861711,
   8324: 8861712,
   7851: 8861713,
   7877: 8861715,
   8212: 8861717,
   8201: 8861719,
   1: 8861714,
   8108: 8861722,
   7902: 8861724,
   8068: 8861726,
   8067: 8861728,
   8032: 8861729,
   7976: 8861731,
   8121: 8861733,
   8115: 8861735,
   7964: 8861736,
   7963: 8861738,
   7429: 8861739,
   7470: 8861741,
   6491: 8861743,
   6515: 8861745,
   7847: 8861747,
   7841: 8861749,
   7816: 8861751,
   7807: 8861753,
   7837: 8861755,
   7836: 8861757,
   7464: 8861758,
   7714: 8861760,
   7683: 8861762,
   7717: 8861764,
   7682: 8861766,
   7572: 8861768,
   7591: 8861770,
   7546: 8861772,
   7507: 8861774,
   7977: 8861776,
   7402: 8861778,
   7320: 8861780,
   7349: 8861782,
   7350: 8861784,
   7327: 8861785,
   7227: 8861787,
   7242: 8861789,
   7035: 8861791,
   7146: 8861793,
   7180: 8861795,
   7103: 8861797,
   7172: 8861798,
   7096: 8861800,
   72